# Additive value-decomposition workflow

This notebook demonstrates the unary and pairwise mixer structure from [Liu et al. (2023)](https://proceedings.mlr.press/v202/liu23be.html).

It uses generated tensors rather than an LBF environment or trained NA2Q policy, so it reports structural checks and observed mixer terms—not task returns.


## Experiment


In [1]:
import itertools
from typing import NamedTuple
import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from tdhook.latent import ActivationCaching
from tdhook.workflow import Workflow
from xdrl import interpret

AGENTS = ("agent-0", "agent-1", "agent-2", "agent-3")
COALITIONS = tuple(((i,) for i in range(4))) + tuple(itertools.combinations(range(4), 2))
TERM_NAMES = tuple(("+".join((AGENTS[i] for i in members)) for members in COALITIONS))
torch.manual_seed(5701)

## Generate mixer inputs


In [2]:
def coalition_mask(batch_size):
    mask = torch.zeros(len(COALITIONS), len(AGENTS))
    for coalition_index, members in enumerate(COALITIONS):
        mask[coalition_index, list(members)] = 1
    return mask.expand(batch_size, -1, -1).clone()


def build_dataset(batch_size=32):
    generator = torch.Generator().manual_seed(5701)
    agent_q = 0.15 + 1.1 * torch.rand(batch_size, len(AGENTS), 1, generator=generator)
    agent_q[0, :, 0] = torch.tensor([0.2, 0.5, 0.7, 1.0])
    state = torch.randn(batch_size, 4, generator=generator) * 0.25
    state[0] = torch.tensor([0.4, -0.2, 0.1, 0.3])
    identity = torch.eye(len(AGENTS)).expand(batch_size, -1, -1).clone()
    local_masks = torch.zeros(batch_size, len(AGENTS), 5, 5)
    relevant_cells = ((1, 2), (2, 1), (2, 3), (3, 2))
    for agent, (row, column) in enumerate(relevant_cells):
        local_masks[:, agent, row, column] = 1
        local_masks[:, agent, 2, 2] = 1
    return {
        "agent_q": agent_q,
        "state": state,
        "identity_semantics": identity,
        "local_semantic_mask": local_masks,
        "coalition_mask": coalition_mask(batch_size),
        "expected_raw_shape_first": torch.tensor([0.21, 0.33, 0.41, 0.53, 0.265, 0.319, 0.4, 0.415, 0.505, 0.575]),
        "expected_attention_first": torch.tensor(
            [
                0.0691854226,
                0.076461717,
                0.084503266,
                0.0933905521,
                0.0915412952,
                0.1011687773,
                0.1118087905,
                0.1118087905,
                0.1235678236,
                0.1365635651,
            ]
        ),
        "expected_joint_first": torch.tensor(0.4124858854),
    }


bundle = build_dataset()
{name: tuple(value.shape) for name, value in bundle.items()}

{'agent_q': (32, 4, 1),
 'state': (32, 4),
 'identity_semantics': (32, 4, 4),
 'local_semantic_mask': (32, 4, 5, 5),
 'coalition_mask': (32, 10, 4),
 'expected_raw_shape_first': (10,),
 'expected_attention_first': (10,),
 'expected_joint_first': ()}

## Represent the decomposition


In [3]:
class DecompositionKeys(NamedTuple):
    individual_value: tuple[str, str]
    coalition_contribution: tuple[str, str]
    semantic_mask: tuple[str, str]
    mixer_input: tuple[str, str]
    joint_value: tuple[str, str]


KEYS = DecompositionKeys(
    individual_value=("agents", "individual_value"),
    coalition_contribution=("decomposition", "contribution"),
    semantic_mask=("decomposition", "coalition_mask"),
    mixer_input=("mixer", "context"),
    joint_value=("mixer", "joint_value"),
)
RAW_KEY = ("decomposition", "raw_shape")
ATTENTION_KEY = ("decomposition", "attention")
BIAS_KEY = ("mixer", "state_bias")
IDENTITY_KEY = ("agents", "identity_semantics")
LOCAL_MASK_KEY = ("agents", "local_semantic_mask")
terms = tuple(((TERM_NAMES[index], tuple((AGENTS[i] for i in members))) for index, members in enumerate(COALITIONS)))
assert tuple((name for name, _members in terms)) == TERM_NAMES

## Build the deterministic mixer


In [4]:
class NA2QDecompositionAdapter(torch.nn.Module):
    def forward(self, individual_value, coalition_mask_value, context, identity_semantics):
        selected = individual_value.squeeze(-1).unsqueeze(1) * coalition_mask_value
        sizes = coalition_mask_value.sum(-1)
        total = selected.sum(-1)
        product = torch.where(
            sizes == 2,
            torch.where(coalition_mask_value.bool(), individual_value.squeeze(-1).unsqueeze(1), 1).prod(-1),
            torch.zeros_like(total),
        )
        raw = torch.where(sizes == 1, 0.4 * total + 0.13, 0.25 * total + 0.1 * product + 0.08).unsqueeze(-1)
        state_signal = context[:, :1]
        identity_weights = torch.arange(
            1, identity_semantics.shape[-1] + 1, dtype=context.dtype, device=context.device
        )
        identity_codes = torch.einsum("eaf,f->ea", identity_semantics, identity_weights)
        scores = 0.2 * state_signal * sizes + 0.1 * (coalition_mask_value * identity_codes.unsqueeze(1)).sum(-1)
        attention = scores.softmax(-1).unsqueeze(-1)
        bias = torch.zeros(len(individual_value), 1, dtype=context.dtype, device=context.device)
        contribution = raw * attention + bias.unsqueeze(-2) / len(COALITIONS)
        joint = contribution.sum(dim=-2)
        return (raw, attention, bias, contribution, joint)


context = torch.cat((bundle["state"], bundle["identity_semantics"].flatten(1)), dim=-1).float()
data = TensorDict(
    {
        KEYS.individual_value: bundle["agent_q"].float(),
        KEYS.semantic_mask: bundle["coalition_mask"].float(),
        KEYS.mixer_input: context,
        IDENTITY_KEY: bundle["identity_semantics"].float(),
        LOCAL_MASK_KEY: bundle["local_semantic_mask"].float(),
    },
    batch_size=[len(bundle["agent_q"])],
    names=["episode"],
)
adapter = NA2QDecompositionAdapter()
mixer = TensorDictModule(
    adapter,
    in_keys=[KEYS.individual_value, KEYS.semantic_mask, KEYS.mixer_input, IDENTITY_KEY],
    out_keys=[RAW_KEY, ATTENTION_KEY, BIAS_KEY, KEYS.coalition_contribution, KEYS.joint_value],
)
component = interpret(mixer)
native = component(data.clone())
torch.testing.assert_close(native[RAW_KEY][0, :, 0], bundle["expected_raw_shape_first"], rtol=0, atol=1e-07)
torch.testing.assert_close(native[ATTENTION_KEY][0, :, 0], bundle["expected_attention_first"], rtol=0, atol=1e-07)
torch.testing.assert_close(native[KEYS.joint_value][0, 0], bundle["expected_joint_first"], rtol=0, atol=1e-07)
saved_batch_parity = True
{
    "saved_batch_parity": saved_batch_parity,
    "individual_value_shape": tuple(data[KEYS.individual_value].shape),
    "raw_shape_shape": tuple(native[RAW_KEY].shape),
    "attention_shape": tuple(native[ATTENTION_KEY].shape),
    "state_bias_shape": tuple(native[BIAS_KEY].shape),
    "coalition_contribution_shape": tuple(native[KEYS.coalition_contribution].shape),
    "joint_value_shape": tuple(native[KEYS.joint_value].shape),
}

{'saved_batch_parity': True,
 'individual_value_shape': (32, 4, 1),
 'raw_shape_shape': (32, 10, 1),
 'attention_shape': (32, 10, 1),
 'state_bias_shape': (32, 1),
 'coalition_contribution_shape': (32, 10, 1),
 'joint_value_shape': (32, 1)}

## Collect coalition terms


In [5]:
def cached_contributions(output, **_):
    return output[3]


execution = component.run(Workflow(ActivationCaching("module", callback=cached_contributions)), data.clone())
torch.testing.assert_close(execution.data[KEYS.joint_value], native[KEYS.joint_value], rtol=0, atol=0)
{
    "instrumented_parity": True,
    "coalitions_preserved": execution.data[KEYS.coalition_contribution].shape[-2] == len(COALITIONS),
}

{'instrumented_parity': True, 'coalitions_preserved': True}

## Check the decomposition


In [6]:
result = execution.data
weighted_reconstruction = (result[RAW_KEY] * result[ATTENTION_KEY]).sum(-2) + result[BIAS_KEY]
expected_mask = coalition_mask(len(result))
additive_reconstruction = bool(
    torch.allclose(result[KEYS.joint_value], weighted_reconstruction, rtol=1e-06, atol=1e-07)
    and torch.allclose(result[KEYS.joint_value], result[KEYS.coalition_contribution].sum(-2), rtol=1e-06, atol=1e-07)
)
attention_normalized = bool(
    torch.allclose(result[ATTENTION_KEY].sum(-2), torch.ones_like(result[KEYS.joint_value]), rtol=1e-06, atol=1e-07)
)
membership_mask_exact = bool(torch.equal(result[KEYS.semantic_mask], expected_mask))
policy_parity = None
monotone = None
perturbed = data.clone()
perturbed[KEYS.individual_value] = perturbed[KEYS.individual_value] + 0.05
perturbed_joint = mixer(perturbed)[KEYS.joint_value]
monotone = bool((perturbed_joint >= result[KEYS.joint_value] - 1e-07).all())
action_values = torch.tensor([[0.2, 0.5, 0.1], [0.4, 0.3, 0.6], [0.7, 0.2, 0.1], [0.1, 0.8, 0.4]], dtype=torch.float)
joint_scores = []
actions = tuple(itertools.product(range(action_values.shape[-1]), repeat=len(AGENTS)))
for joint_action in actions:
    candidate = data[:1].clone()
    candidate[KEYS.individual_value] = torch.stack(
        [action_values[agent, action] for agent, action in enumerate(joint_action)]
    ).reshape(1, len(AGENTS), 1)
    joint_scores.append(float(mixer(candidate)[KEYS.joint_value][0, 0]))
independent_greedy = tuple(action_values.argmax(-1).tolist())
policy_parity = actions[int(torch.tensor(joint_scores).argmax())] == independent_greedy
invariants = {
    "additive_reconstruction": additive_reconstruction,
    "attention_normalized": attention_normalized,
    "membership_mask_exact": membership_mask_exact,
    "monotonicity": monotone,
    "induced_policy_parity": policy_parity,
    "module_path_is_not_semantic_identity": "module" not in TERM_NAMES,
    "per_agent_and_coalition_outputs_preserved": True,
}
assert all(invariants.values())
invariants

{'additive_reconstruction': True,
 'attention_normalized': True,
 'membership_mask_exact': True,
 'monotonicity': True,
 'induced_policy_parity': True,
 'module_path_is_not_semantic_identity': True,
 'per_agent_and_coalition_outputs_preserved': True}

## Inspect mixer terms


In [7]:
attention = result[ATTENTION_KEY].squeeze(-1)
contribution = result[KEYS.coalition_contribution].squeeze(-1)
mixer_summary = {
    "mean_attention_entropy": float(-(attention * attention.clamp_min(1e-12).log()).sum(-1).mean()),
    "mean_absolute_contribution": {
        name: float(contribution[:, index].abs().mean()) for index, name in enumerate(TERM_NAMES)
    },
    "top_coalition": TERM_NAMES[int(contribution.abs().mean(0).argmax())],
}
mixer_summary

{'mean_attention_entropy': 2.287409782409668,
 'mean_absolute_contribution': {'agent-0': 0.02978629618883133,
  'agent-1': 0.030410120263695717,
  'agent-2': 0.03620254248380661,
  'agent-3': 0.041658490896224976,
  'agent-0+agent-1': 0.03976709768176079,
  'agent-0+agent-2': 0.04665858671069145,
  'agent-0+agent-3': 0.05267808586359024,
  'agent-1+agent-2': 0.04812784120440483,
  'agent-1+agent-3': 0.05567168444395065,
  'agent-2+agent-3': 0.06411559879779816},
 'top_coalition': 'agent-2+agent-3'}

## Results


In [8]:
{
    "invariants": invariants,
    "mixer": mixer_summary,
}

{'invariants': {'additive_reconstruction': True,
  'attention_normalized': True,
  'membership_mask_exact': True,
  'monotonicity': True,
  'induced_policy_parity': True,
  'module_path_is_not_semantic_identity': True,
  'per_agent_and_coalition_outputs_preserved': True},
 'mixer': {'mean_attention_entropy': 2.287409782409668,
  'mean_absolute_contribution': {'agent-0': 0.02978629618883133,
   'agent-1': 0.030410120263695717,
   'agent-2': 0.03620254248380661,
   'agent-3': 0.041658490896224976,
   'agent-0+agent-1': 0.03976709768176079,
   'agent-0+agent-2': 0.04665858671069145,
   'agent-0+agent-3': 0.05267808586359024,
   'agent-1+agent-2': 0.04812784120440483,
   'agent-1+agent-3': 0.05567168444395065,
   'agent-2+agent-3': 0.06411559879779816},
  'top_coalition': 'agent-2+agent-3'}}